# JOUR328 Assignment - Elevating a Story with Data

I chose this <a href="https://dbknews.com/2025/05/12/umd-researchers-nsf-trump-grants/">Diamondback story</a> published in May 2025, becuase there's no graph in it but they actually wrote the story based on a comprehensive dataset from <a href ="https://grant-witness.us/nsf-data.html">Grant Witness</a>.

In [7]:
import pandas as pd

df = pd.read_csv("nsf_terminations.csv")
df["status"] = df["status"].str.replace(r"^[^\w]+\s*", "", regex=True)
print(f"Total rows: {len(df)}")
df.head(3)

Total rows: 1996


,grant_id,status,terminated,suspended,termination_date,termination_indicator,reinstated,reinstatement_date,reinstatement_indicator,cruz_list,...,usasp_outlaid,estimated_budget,estimated_outlays,estimated_remaining,post_termination_deobligation,division,directorate,div,dir,record_sha1
0,2201796,Terminated,True,False,2025-04-18,"Self reported, Added to NSF CSV, research.gov",False,NaN,NaN,True,...,969292.90,1083218,969292.90,113925.10,-113925.0,Research on Learning in Formal and Informal Se...,STEM Education,DRL,EDU,0ebc9c03ac9d04ff28cf22b918a86c9c7a54a920
1,2119573,Terminated,True,False,2025-04-25,"Self reported, Added to NSF CSV, research.gov",False,NaN,NaN,False,...,529355.37,582983,529355.37,53627.63,NaN,Information and Intelligent Systems,Computer and Information Science and Engineering,IIS,CISE,a1bf208e2311dbea539a493767e76be6ca810042
2,2139096,Terminated,True,False,2025-04-25,"Self reported, Added to NSF CSV, research.gov",False,NaN,NaN,False,...,268998.65,454335,268998.65,185336.35,NaN,Engineering Education and Centers,Engineering,EEC,ENG,bb3274bc3ce11110f0826e5aa47e77b8e8e5064f


## Filter: Truly Terminated Grants

We exclude grants marked as "Possibly Reinstated" and keep only confirmed terminations.

In [8]:
# Filter to truly terminated grants only (exclude "Possibly Reinstated")
terminated = df[df["status"] == "Terminated"].copy()

print(f"Truly terminated:     {len(terminated)}")
print(f"Possibly reinstated:  {len(df) - len(terminated)}")
print(f"Total:                {len(df)}")

# Clean up directorate: treat blank/NA as "(Unknown)"
terminated["directorate"] = terminated["directorate"].fillna("(Unknown)").replace("NA", "(Unknown)").replace("", "(Unknown)")

# Ensure financial columns are numeric
for col in ["estimated_budget", "estimated_outlays", "estimated_remaining"]:
    terminated[col] = pd.to_numeric(terminated[col], errors="coerce").fillna(0)

Truly terminated:     1363
Possibly reinstated:  633
Total:                1996


## Aggregate by Directorate

In [11]:
# Group by directorate and division, then aggregate
by_directorate = (
    terminated
    .groupby(["directorate", "division"], as_index=False)
    .agg(
        grants=("grant_id", "count"),
        estimated_budget=("estimated_budget", "sum"),
        estimated_outlays=("estimated_outlays", "sum"),
        estimated_remaining=("estimated_remaining", "sum"),
    )
    .sort_values("estimated_budget", ascending=False)
    .reset_index(drop=True)
)

# Add a totals row
totals = pd.DataFrame([{
    "directorate": "TOTAL",
    "division": "",
    "grants": by_directorate["grants"].sum(),
    "estimated_budget": by_directorate["estimated_budget"].sum(),
    "estimated_outlays": by_directorate["estimated_outlays"].sum(),
    "estimated_remaining": by_directorate["estimated_remaining"].sum(),
}])
by_directorate = pd.concat([by_directorate, totals], ignore_index=True)

# Format for display
display_df = by_directorate.copy()
for col in ["estimated_budget", "estimated_outlays", "estimated_remaining"]:
    display_df[col] = display_df[col].apply(lambda x: f"${x:,.0f}")

display_df.columns = ["Directorate", "Division", "Grants", "Est. Budget", "Est. Outlays", "Est. Remaining"]
display_df

,Directorate,Division,Grants,Est. Budget,Est. Outlays,Est. Remaining
0,STEM Education,Equity for Excellence in STEM,436,"$618,678,198","$280,220,942","$338,457,256"
1,STEM Education,Research on Learning in Formal and Informal Se...,211,"$248,241,906","$110,962,986","$137,278,920"
2,"Technology, Innovation and Partnerships","Technology, Innovation and Partnerships",29,"$49,156,117","$27,367,414","$21,788,703"
3,"Social, Behavioral and Economic Sciences",Social and Economic Sciences,142,"$45,421,107","$26,252,262","$19,168,845"
4,Geosciences,"Research, Innovation, Synergies, and Education",41,"$44,538,264","$13,403,386","$31,134,878"
5,STEM Education,Graduate Education,41,"$38,497,499","$15,115,696","$23,381,803"
6,STEM Education,Undergraduate Education,49,"$36,487,729","$14,659,596","$21,828,133"
7,Computer and Information Science and Engineering,Computer and Network Systems,53,"$32,340,598","$20,577,081","$11,763,517"
8,Engineering,Engineering Education and Centers,71,"$31,905,156","$14,524,676","$17,380,480"
9,Biological Sciences,Biological Infrastructure,36,"$24,910,322","$12,384,746","$12,525,576"


In [13]:
display_df.to_csv("nsf_grap01.csv")